# Bulk Read with Wildcards
construct a robust read command that dynamically loads all 92 daily CSV files without failing on data types.

In [0]:
# Define your exact workspace path
workspace_path = "/Workspace/Shared/DS625_Team_Z/backblaze_drive_data_Q4_2025"

# Load all 92 daily CSV files from the shared directory
raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{workspace_path}/*.csv")

print(f"Success! Total rows loaded from Q4 2025: {raw_df.count()}")

Success! Total rows loaded from Q4 2025: 30941708


#  Target Column Selection & Type Safety
The Backblaze schema has over 100 columns. Filter only for the vital identifiers and the 6 universally predictive SMART raw attributes.

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import LongType

smart_cols = [
    'smart_5_raw', 'smart_9_raw', 'smart_187_raw',
    'smart_188_raw', 'smart_197_raw', 'smart_198_raw'
]

selected_cols = [
    'date', 'serial_number', 'model', 'capacity_bytes', 'failure'
] + smart_cols

clean_df = raw_df.select(
    *selected_cols
)

for c in smart_cols:
    clean_df = clean_df.withColumn(c, col(c).cast(LongType()))

clean_df = clean_df.fillna(0, subset=smart_cols)

display(clean_df)

date,serial_number,model,capacity_bytes,failure,smart_5_raw,smart_9_raw,smart_187_raw,smart_188_raw,smart_197_raw,smart_198_raw
2025-12-23,2206E608DB42,CT250MX500SSD1,250059350016,0,0,2312,0,0,0,0
2025-12-23,2207E60CC65A,CT250MX500SSD1,250059350016,0,0,25373,0,0,0,0
2025-12-23,2340E87B92B5,CT250MX500SSD1,250059350016,0,0,11926,0,0,0,0
2025-12-23,2340E87B97E8,CT250MX500SSD1,250059350016,0,0,7031,0,0,0,0
2025-12-23,2407E896B6D5,CT250MX500SSD1,250059350016,0,0,3199,0,0,0,0
2025-12-23,2EGK64VX,HGST HUH728080ALE604,8001563222016,0,0,41082,0,0,0,0
2025-12-23,2EHZAKAX,HGST HUH728080ALE604,8001563222016,0,0,79685,0,0,0,0
2025-12-23,2EJ02A1X,HGST HUH728080ALE604,8001563222016,0,0,79734,0,0,0,0
2025-12-23,7LZ021LA,Seagate BarraCuda SSD ZA250CM10002,250059350016,0,0,51084,0,0,0,0
2025-12-23,S2ZYJ9CF511681,ST500LM012 HN,500107862016,0,0,78798,0,0,0,0


## Forward-Looking Label Engineering

A drive's target label shouldn't just flag the single day it died; it needs to look ahead 30 days so your machine learning model learns to predict upcoming failure.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import max as spark_max, col

w = Window.partitionBy("serial_number").orderBy("date").rowsBetween(0, 30)

labeled_df = clean_df.withColumn(
    "label",
    spark_max(col("failure")).over(w)
)

display(labeled_df)

date,serial_number,model,capacity_bytes,failure,smart_5_raw,smart_9_raw,smart_187_raw,smart_188_raw,smart_197_raw,smart_198_raw,label
2025-10-01,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-02,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-03,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-04,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-05,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-06,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-07,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-08,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-09,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0
2025-10-10,000a43e7dee60010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0


# Write Optimization for the Free Tier
Save the engineered data into Delta format so notebooks 02 and 03 can read it in milliseconds without re-processing the raw text.

In [0]:
# Save directly as a managed table without calling .cache() or .persist()
labeled_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("backblaze_master_delta")

print("Master dataset successfully saved to serverless storage!")

Master dataset successfully saved to serverless storage!


# How to Load This in Notebooks 02 and 03
Once that cell runs cleanly, the table is locked into your serverless workspace environment. You can call it into your EDA and Feature Engineering notebooks using a single line of code:
```
# Run this at the top of your next notebooks
df = spark.table("backblaze_master_delta")
```